# Generative Adversarial Network (GAN) from Scratch
## DCGAN for Image Generation

GANs train two networks in competition:
- **Generator**: Creates fake images from random noise
- **Discriminator**: Distinguishes real from fake

Through adversarial training, the generator learns to create increasingly realistic images.

---

In [ ]:
# CELL 1: Setup
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as transforms
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
BATCH_SIZE = 128; EPOCHS = 50; NZ = 100; NGF = 64; NDF = 64; LR = 0.0002

transform = transforms.Compose([transforms.Resize(64), transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])])
train_data = torchvision.datasets.MNIST('data', train=True, download=True, transform=transform)
loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
print(f'Device: {device} | Samples: {len(train_data)}')

In [ ]:
# CELL 2: Generator Network (DCGAN Architecture)
class Generator(nn.Module):
    """
    DCGAN Generator: Maps random noise z -> image.
    Uses transposed convolutions to upsample:
    z (100) -> 4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64
    """
    def __init__(self, nz=100, ngf=64, nc=1):
        super().__init__()
        self.main = nn.Sequential(
            # Input: z (nz x 1 x 1)
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),      # -> ngf*8 x 4 x 4
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),      # -> ngf*4 x 8 x 8
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),      # -> ngf*2 x 16 x 16
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),         # -> ngf x 32 x 32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()                                    # -> nc x 64 x 64
        )
    
    def forward(self, z):
        return self.main(z)

class Discriminator(nn.Module):
    """
    DCGAN Discriminator: Maps image -> real/fake probability.
    Mirror of Generator: 64x64 -> 32x32 -> ... -> scalar
    """
    def __init__(self, nc=1, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),                    # -> ndf x 32 x 32
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True), # -> ndf*2 x 16 x 16
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, True), # -> ndf*4 x 8 x 8
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, True), # -> ndf*8 x 4 x 4
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()                                 # -> 1 x 1 x 1
        )
    
    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)

# Weight initialization (from DCGAN paper)
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.constant_(m.bias, 0)

netG = Generator(NZ, NGF).to(device).apply(weights_init)
netD = Discriminator(1, NDF).to(device).apply(weights_init)
print(f"Generator params: {sum(p.numel() for p in netG.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in netD.parameters()):,}")

In [ ]:
# CELL 3: GAN Training Loop
criterion = nn.BCELoss()
optimD = optim.Adam(netD.parameters(), lr=LR, betas=(0.5, 0.999))
optimG = optim.Adam(netG.parameters(), lr=LR, betas=(0.5, 0.999))
fixed_noise = torch.randn(64, NZ, 1, 1, device=device)

G_losses, D_losses = [], []
for epoch in range(EPOCHS):
    for real_imgs, _ in loader:
        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)
        real_label = torch.ones(b_size, device=device) * 0.9  # Label smoothing
        fake_label = torch.zeros(b_size, device=device)
        
        # === Train Discriminator ===
        netD.zero_grad()
        output_real = netD(real_imgs)
        lossD_real = criterion(output_real, real_label)
        
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, fake_label)
        
        lossD = lossD_real + lossD_fake
        lossD.backward(); optimD.step()
        
        # === Train Generator ===
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, torch.ones(b_size, device=device))
        lossG.backward(); optimG.step()
    
    G_losses.append(lossG.item()); D_losses.append(lossD.item())
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] D_loss: {lossD.item():.4f} G_loss: {lossG.item():.4f}")

plt.plot(G_losses, label='Generator'); plt.plot(D_losses, label='Discriminator')
plt.title('GAN Training Losses'); plt.legend()
plt.savefig(OUTPUT_DIR / 'gan_training.png', dpi=150); plt.show()

In [ ]:
# CELL 4: Generate & Visualize Final Images
netG.eval()
with torch.no_grad():
    fake_images = netG(fixed_noise).cpu()

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(fake_images[i].squeeze() * 0.5 + 0.5, cmap='gray')
    ax.axis('off')
plt.suptitle('DCGAN Generated Digits (Epoch {})'.format(EPOCHS), fontsize=14)
plt.savefig(OUTPUT_DIR / 'gan_generated.png', dpi=150); plt.show()

# Save models
torch.save(netG.state_dict(), OUTPUT_DIR / 'generator.pt')
torch.save(netD.state_dict(), OUTPUT_DIR / 'discriminator.pt')
print("\nGAN complete! All 4 deep learning models built.")
print("Models saved to outputs/")